In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [ ]:
%%writefile hello.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void helloFromGPU()
{
    printf("Hello World from GPU! Thread %d\n", threadIdx.x);
}

int main()
{
    printf("Hello World from CPU!\n");

    helloFromGPU<<<1, 10>>>();

    cudaError_t err = cudaGetLastError();
    if (err != cudaSuccess) {
        printf("Kernel launch error: %s\n", cudaGetErrorString(err));
        return 1;
    }

    err = cudaDeviceSynchronize();
    if (err != cudaSuccess) {
        printf("GPU execution error: %s\n", cudaGetErrorString(err));
        return 1;
    }

    cudaDeviceReset();
    return 0;
}


Overwriting hello.cu


In [ ]:
!nvcc hello.cu -o hello && ./hello

Hello World from CPU!
Hello World from GPU! Thread 0
Hello World from GPU! Thread 1
Hello World from GPU! Thread 2
Hello World from GPU! Thread 3
Hello World from GPU! Thread 4
Hello World from GPU! Thread 5
Hello World from GPU! Thread 6
Hello World from GPU! Thread 7
Hello World from GPU! Thread 8
Hello World from GPU! Thread 9


Write a CUDA C++ program to add two arrays of \(1,000,000\) integers using both the CPU and GPU. Initialize the first array with values \(A[i]=i\) and the second array with values \(B[i]=2i\).


In [ ]:

%%writefile lab11.cu
#include <stdio.h>
#include <cuda_runtime.h>
#include <chrono>

#define N 1000000

// GPU function
__global__ void add(int *a, int *b, int *c)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < N)
        c[i] = a[i] + b[i];
}

int main()
{
    int *a, *b, *c;
    int *da, *db, *dc;

    a = new int[N];
    b = new int[N];
    c = new int[N];

    for (int i = 0; i < N; i++)
    {
        a[i] = i;
        b[i] = i * 2;
    }

    // CPU addition
    auto start = std::chrono::high_resolution_clock::now();

    for (int i = 0; i < N; i++)
        c[i] = a[i] + b[i];

    auto stop = std::chrono::high_resolution_clock::now();

    double cpuTime =
        std::chrono::duration<double, std::milli>(
            stop - start).count();

    cudaMalloc(&da, N * sizeof(int));
    cudaMalloc(&db, N * sizeof(int));
    cudaMalloc(&dc, N * sizeof(int));

    cudaMemcpy(da, a, N * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(db, b, N * sizeof(int), cudaMemcpyHostToDevice);

    // GPU addition
    int threads = 256;
    int blocks = (N + threads - 1) / threads;

    auto gpuStart = std::chrono::high_resolution_clock::now();

    add<<<blocks, threads>>>(da, db, dc);
    cudaDeviceSynchronize();

    auto gpuStop = std::chrono::high_resolution_clock::now();

    double gpuTime =
        std::chrono::duration<double, std::milli>(
            gpuStop - gpuStart).count();

    cudaMemcpy(c, dc, N * sizeof(int), cudaMemcpyDeviceToHost);

    // Check results
    bool correct = true;

    for (int i = 0; i < N; i++)
    {
        if (c[i] != a[i] + b[i])
        {
            correct = false;
            break;
        }
    }

    printf("CPU time: %.4f ms\n", cpuTime);
    printf("GPU time: %.4f ms\n", gpuTime);
    printf("Results correct: %s\n", correct ? "Yes" : "No");

    printf("First five results:\n");

    for (int i = 0; i < 5; i++)
        printf("%d + %d = %d\n", a[i], b[i], c[i]);

    printf("Speedup: %.2fx\n", cpuTime / gpuTime);

    cudaFree(da);
    cudaFree(db);
    cudaFree(dc);

    delete[] a;
    delete[] b;
    delete[] c;

    return 0;
}


Writing lab11.cu


In [ ]:
!nvcc -O2 lab11.cu -o lab11
!./lab11

CPU time: 1.7867 ms
GPU time: 0.1790 ms
Results correct: Yes
First five results:
0 + 0 = 0
1 + 2 = 3
2 + 4 = 6
3 + 6 = 9
4 + 8 = 12
Speedup: 9.98x


Write a CUDA C++ program to add two \(4 \times 4\) matrices using a GPU. Each thread should calculate one element of the resultant matrix. Use a two-dimensional grid of threads, copy the matrices to GPU memory, and display the resulting matrix.

In [ ]:

%%writefile matrix_add.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define N 4

// GPU function
__global__ void matrixAdd(int *a, int *b, int *c)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N)
    {
        int i = row * N + col;
        c[i] = a[i] + b[i];
    }
}

int main()
{
    int a[N][N], b[N][N], c[N][N];
    int *da, *db, *dc;

    // Initialize matrices
    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
        {
            a[i][j] = i * N + j + 1;
            b[i][j] = i + 1;
        }
    }

    // Allocate GPU memory
    cudaMalloc(&da, N * N * sizeof(int));
    cudaMalloc(&db, N * N * sizeof(int));
    cudaMalloc(&dc, N * N * sizeof(int));

    // Copy matrices to GPU
    cudaMemcpy(da, a, N * N * sizeof(int),
               cudaMemcpyHostToDevice);
    cudaMemcpy(db, b, N * N * sizeof(int),
               cudaMemcpyHostToDevice);

    // Configure threads and blocks
    dim3 threads(4, 4);
    dim3 blocks(1, 1);

    // Execute kernel
    matrixAdd<<<blocks, threads>>>(da, db, dc);
    cudaDeviceSynchronize();

    // Copy result to CPU
    cudaMemcpy(c, dc, N * N * sizeof(int),
               cudaMemcpyDeviceToHost);

    // Display result
    printf("Result Matrix:\n");

    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
        {
            printf("%d ", c[i][j]);
        }
        printf("\n");
    }

    // Free memory
    cudaFree(da);
    cudaFree(db);
    cudaFree(dc);

    return 0;
}


Writing matrix_add.cu


In [ ]:
!nvcc -O2 matrix_add.cu -o matrix_add
!./matrix_add

Result Matrix:
2 3 4 5 
7 8 9 10 
12 13 14 15 
17 18 19 20 


Transpose a matrix using CUDA, where rows become columns and columns become rows.

In [ ]:

%%writefile matrix_transpose.cu
#include <stdio.h>
#include <cuda_runtime.h>

#define N 3

__global__ void transpose(int *a, int *b)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N)
    {
        b[col * N + row] = a[row * N + col];
    }
}

int main()
{
    int a[N][N] = {
        {1, 2, 3},
        {4, 5, 6},
        {7, 8, 9}
    };

    int b[N][N];
    int *da, *db;

    cudaMalloc(&da, N * N * sizeof(int));
    cudaMalloc(&db, N * N * sizeof(int));

    cudaMemcpy(da, a, N * N * sizeof(int),
               cudaMemcpyHostToDevice);

    dim3 threads(3, 3);
    dim3 blocks(1, 1);

    transpose<<<blocks, threads>>>(da, db);
    cudaDeviceSynchronize();

    cudaMemcpy(b, db, N * N * sizeof(int),
               cudaMemcpyDeviceToHost);

    printf("Transposed Matrix:\n");

    for (int i = 0; i < N; i++)
    {
        for (int j = 0; j < N; j++)
            printf("%d ", b[i][j]);

        printf("\n");
    }

    cudaFree(da);
    cudaFree(db);

    return 0;
}


Writing matrix_transpose.cu


In [ ]:
!nvcc -O2 matrix_transpose.cu -o matrix_transpose
!./matrix_transpose

Transposed Matrix:
1 4 7 
2 5 8 
3 6 9 
